# Assignment 25: Prompting & Langchain Chains

# PART 1 -- Prompt Templates

# Task 1: PromptTemplate
## 1. Create a PromptTemplate with:
## . System instruction
## . User question placeholder
## 2. Inject user input dynamically.
## 3. Test with multiple inputs.

In [5]:
## Creating prompt template
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    input_variables=["question"],
    template="""
    You are a helpful AI assistant.

    System Instruction:
    Answer the user's question in a simple and clear way.
    Keep your answer short and informative.

    User Question:
    {question}

    Answer:
    """
)

In [4]:
## Inject user input dynamically
response = prompt.format(
    question="What is Artificial Intelligence?"
)

print(response)


    You are a helpful AI assistant.

    System Instruction:
    Answer the user's question in a simple and clear way.
    Keep your answer short and informative.

    User Question:
    What is Artificial Intelligence?

    Answer:
    


In [3]:
## Test with multiple inputs
questions = [
    "What is Python?",
    "What is Machine Learning?",
    "What is Generative AI?",
    "What is LangChain?"
]

for question in questions:

    result = prompt.format(question=question)

    print("\nQuestion:", question)
    print("Prompt:", result)


Question: What is Python?
Prompt: 
    You are a helpful AI assistant.

    System Instruction:
    Answer the user's question in a simple and clear way.
    Keep your answer short and informative.

    User Question:
    What is Python?

    Answer:
    

Question: What is Machine Learning?
Prompt: 
    You are a helpful AI assistant.

    System Instruction:
    Answer the user's question in a simple and clear way.
    Keep your answer short and informative.

    User Question:
    What is Machine Learning?

    Answer:
    

Question: What is Generative AI?
Prompt: 
    You are a helpful AI assistant.

    System Instruction:
    Answer the user's question in a simple and clear way.
    Keep your answer short and informative.

    User Question:
    What is Generative AI?

    Answer:
    

Question: What is LangChain?
Prompt: 
    You are a helpful AI assistant.

    System Instruction:
    Answer the user's question in a simple and clear way.
    Keep your answer short and inform

# Task 2: ChatPromptTemplate & Message Templates
## 1. Create a ChatPromptTemplate using
## . systemMessage
## . HumanMessage
## . AIMessage
## 2. Use message prompt templates to structure conversation
## 3. Compare with simple PromptTemplate

In [6]:
## Creating a ChatPromptTemplate
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Answer in simple language."),
    ("human", "{question}"),
    ("ai", "I will explain your question clearly.")
])

In [7]:
## Structure Conversation
messages = chat_prompt.format_messages(
    question="What is Generative AI?"
)

for message in messages:
    print(type(message).__name__, ":", message.content)

SystemMessage : You are a helpful AI assistant. Answer in simple language.
HumanMessage : What is Generative AI?
AIMessage : I will explain your question clearly.


In [8]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage(content="You are a helpful AI assistant."),
    HumanMessage(content="What is Python?"),
    AIMessage(content="Python is a programming language."),
    HumanMessage(content="Where is Python used?")
]

for msg in messages:
    print(msg.type, ":", msg.content)

system : You are a helpful AI assistant.
human : What is Python?
ai : Python is a programming language.
human : Where is Python used?


In [9]:
questions = [
    "What is Machine Learning?",
    "What is Deep Learning?",
    "What is Natural Language Processing?"
]

for question in questions:

    messages = chat_prompt.format_messages(
        question=question
    )

    print("\nQuestion:", question)

    for msg in messages:
        print(msg.type, ":", msg.content)


Question: What is Machine Learning?
system : You are a helpful AI assistant. Answer in simple language.
human : What is Machine Learning?
ai : I will explain your question clearly.

Question: What is Deep Learning?
system : You are a helpful AI assistant. Answer in simple language.
human : What is Deep Learning?
ai : I will explain your question clearly.

Question: What is Natural Language Processing?
system : You are a helpful AI assistant. Answer in simple language.
human : What is Natural Language Processing?
ai : I will explain your question clearly.


## Comparision with Simple PromptTemplate

##  PromptTemplate vs ChatPromptTemplate
###  Feature	PromptTemplate	ChatPromptTemplate
### Structure	Simple text	Multiple messages
### Roles	No message roles	System, Human, AI
### Conversation	Basic	Structured
### Dynamic input	Yes	Yes

# PART 2-- Structured Output using Pydantic

# Task 3: Pydantic Output Schema
## 1. Define a pYdantic model for chatbot response:
## class Answer(BaseModel):
## answer: str
## confidence : float
## source : str
## 2 Use Langchain output parsers
## 3. Ensure LLM output follows the schema.

In [10]:
## Define Pydantic model
from pydantic import BaseModel, Field

class Answer(BaseModel):
    answer: str
    confidence: float
    source: str

In [11]:
## Create a langchain output parser
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(
    pydantic_object=Answer
)

print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"answer": {"title": "Answer", "type": "string"}, "confidence": {"title": "Confidence", "type": "number"}, "source": {"title": "Source", "type": "string"}}, "required": ["answer", "confidence", "source"]}
```


In [12]:
## Create a Prompt Template
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""
    You are a helpful AI chatbot.

    Answer the user's question.

    Question: {question}

    {format_instructions}
    """,
    input_variables=["question"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    }
)

In [13]:
## ensure LLM output follows the schema
import os
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

chain = prompt | llm | parser

response = chain.invoke({
    "question": "What is Machine Learning?"
})

print("Answer:", response.answer)
print("Confidence:", response.confidence)
print("Source:", response.source)

Answer: Machine Learning is a subfield of artificial intelligence that focuses on developing algorithms and statistical models that enable computers to learn from and make predictions or decisions based on data, without being explicitly programmed for each specific task.
Confidence: 0.99
Source: Wikipedia


# Task 4: Validation & Error Handling
## 1. Handle invalid LLM outputs
## 2. Retry or fallback when schema is not followed

In [14]:
invalid_output = """
{
    "answer": "Python is a programming language",
    "confidence": "high"
}
"""

try:
    result = parser.parse(invalid_output)
    print(result)

except Exception as e:
    print("Invalid Output!")
    print("Error:", e)

Invalid Output!
Error: Failed to parse Answer from completion {"answer": "Python is a programming language", "confidence": "high"}. Got: 2 validation errors for Answer
confidence
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='high', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/float_parsing
source
  Field required [type=missing, input_value={'answer': 'Python is a p...', 'confidence': 'high'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 


In [15]:
## Retry machanism
from langchain_core.exceptions import OutputParserException

def validate_output(output, retries=3):

    for attempt in range(retries):

        try:
            result = parser.parse(output)
            return result

        except Exception as e:
            print(f"Attempt {attempt + 1} failed")
            print("Error:", e)

            output = """
            {
                "answer": "Python is a high-level programming language.",
                "confidence": 0.95,
                "source": "General Knowledge"
            }
            """

    return None


result = validate_output(invalid_output)

if result:
    print("\nValid Response:")
    print(result)

Attempt 1 failed
Error: Failed to parse Answer from completion {"answer": "Python is a programming language", "confidence": "high"}. Got: 2 validation errors for Answer
confidence
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='high', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/float_parsing
source
  Field required [type=missing, input_value={'answer': 'Python is a p...', 'confidence': 'high'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 

Valid Response:
answer='Python is a high-level programming language.' confidence=0.95 source='General Knowledge'


In [16]:
## Fallback machenism
def safe_response(output):

    try:
        return parser.parse(output)

    except Exception:
        return Answer(
            answer="Sorry, I could not generate a valid response.",
            confidence=0.0,
            source="Fallback"
        )


result = safe_response(invalid_output)

print("Answer:", result.answer)
print("Confidence:", result.confidence)
print("Source:", result.source)

Answer: Sorry, I could not generate a valid response.
Confidence: 0.0
Source: Fallback


# PART 3 -- Chains in Langchain

# Task 5: Simple Chain
## 1. Create a basic chain:
## Prompt ---> Any LLM ---> Output
## 2. Execute the chain.

In [17]:

from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

# Load API key
load_dotenv()

# 1. Create Prompt Template
prompt = PromptTemplate(
    input_variables=["topic"],
    template="""
    You are a helpful AI assistant.

    Explain the following topic in simple language:

    Topic: {topic}
    """
)

# 2. Initialize LLM
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7
)

# 3. Create Simple Chain
chain = prompt | llm

# 4. Execute Chain
topic = input("Enter a topic: ")

response = chain.invoke({
    "topic": topic
})

# 5. Display Output
print(response.content)


**What is AI? (Explained Simply)**  

**1. The basic idea**  
Artificial Intelligence, or **AI**, is a way of making computers and machines think or act a bit like people. Instead of just following exact step‑by‑step instructions, an AI can learn from data, notice patterns, and make decisions on its own.

**2. How it works (in plain terms)**  

| Human brain | AI (computer) |
|-------------|----------------|
| Learns from experience (seeing, hearing, doing) | Learns from data (pictures, text, numbers) |
| Recognizes faces, understands speech, solves problems | Recognizes faces in photos, transcribes spoken words, suggests the best route on a map |
| Can improve with practice | Improves when it gets more data or feedback |

The “learning” part is usually done with **machine learning**—a set of techniques that let the computer adjust its own rules based on examples it’s given.

**3. Everyday examples you already see**  

| Example | What AI is doing |
|---------|-------------------|
| **

# Task 6: Conditional Chain
## 1. Create a chain that:
## . Uses retrieval if question is factual
## . Uses direct LLM response otherwise
## 2. Implement conditional logic.

In [ ]:

from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch, RunnableLambda

from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Load API key
load_dotenv()

# 1. Initialize LLM
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

# 2. Create sample knowledge base
documents = [
    Document(page_content="Python is a high-level programming language."),
    Document(page_content="Machine Learning is a subset of Artificial Intelligence."),
    Document(page_content="LangChain is a framework for building LLM applications."),
    Document(page_content="RAG stands for Retrieval Augmented Generation.")
]

# 3. Create embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# 4. Create Vector Database
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

# 5. Create Prompts
retrieval_prompt = ChatPromptTemplate.from_template("""
Answer the question using the following context.

Context:
{context}

Question:
{question}

If the answer is not available in the context,
say that the information is not available.
""")

direct_prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant.

Answer the following question naturally:

{question}
""")

# 6. Create Retrieval Chain
retrieval_chain = (
    RunnableLambda(
        lambda x: {
            "context": "\n".join(
                doc.page_content
                for doc in retriever.invoke(x["question"])
            ),
            "question": x["question"]
        }
    )
    | retrieval_prompt
    | llm
    | StrOutputParser()
)

# 7. Create Direct LLM Chain
direct_chain = (
    direct_prompt
    | llm
    | StrOutputParser()
)

# 8. Define Conditional Logic
def is_factual(inputs):
    question = inputs["question"].lower()

    factual_keywords = [
        "what is",
        "who is",
        "when was",
        "where is",
        "define",
        "explain",
        "how does"
    ]

    return any(
        keyword in question
        for keyword in factual_keywords
    )

# 9. Create Conditional Chain
conditional_chain = RunnableBranch(
    (
        is_factual,
        retrieval_chain
    ),
    direct_chain
)

# 10. Execute Chain
if __name__ == "__main__":

    while True:
        question = input("\nEnter your question (type exit to stop): ")

        if question.lower() == "exit":
            break

        result = conditional_chain.invoke({
            "question": question
        })
        print(result)


C:\Users\Anish Rawat\AppData\Local\Temp\ipykernel_17760\2810266680.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma
C:\Users\Anish Rawat\AppData\Local\Temp\ipykernel_17760\2810266680.py:30: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
c:\Users\Anish Rawat\GEN_AI\Langchain Projects\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See h

# Task 7: Parallel Chain
## 1. Run Multiple chains in parallel:
## . One chain generates answer
## . One chain generates summary
## . One chain generates follow-up questions
## 2. Combine outputs

In [1]:

from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

# Load API key
load_dotenv()

# 1. Initialize LLM
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7
)

# 2. Create Answer Prompt
answer_prompt = PromptTemplate(
    input_variables=["question"],
    template="""
    Answer the following question in detail:

    {question}
    """
)

# 3. Create Summary Prompt
summary_prompt = PromptTemplate(
    input_variables=["question"],
    template="""
    Give a short summary of the following topic:

    {question}
    """
)

# 4. Create Follow-up Questions Prompt
followup_prompt = PromptTemplate(
    input_variables=["question"],
    template="""
    Generate 3 useful follow-up questions
    related to this topic:

    {question}
    """
)

# 5. Create Individual Chains
answer_chain = (
    answer_prompt
    | llm
    | StrOutputParser()
)

summary_chain = (
    summary_prompt
    | llm
    | StrOutputParser()
)

followup_chain = (
    followup_prompt
    | llm
    | StrOutputParser()
)

# 6. Run All Chains in Parallel
parallel_chain = RunnableParallel(
    answer=answer_chain,
    summary=summary_chain,
    followup_questions=followup_chain
)

# 7. Execute the Parallel Chain
if __name__ == "__main__":

    question = input("Enter your question: ")

    result = parallel_chain.invoke({
        "question": question
    })

    # 8. Display Combined Output

    print("\n1. ANSWER:")
    print(result["answer"])

    print("\n2. SUMMARY:")
    print(result["summary"])

    print("\n3. FOLLOW-UP QUESTIONS:")
    print(result["followup_questions"])



1. ANSWER:
**Python – An In‑Depth Overview**

---

## 1. What Is Python?

Python is a high‑level, interpreted, general‑purpose programming language. It was created by **Guido van Rossum** and first released in **1991**. Python’s design philosophy emphasizes readability, simplicity, and explicitness, making it one of the most popular languages for beginners and seasoned developers alike.

Key characteristics:

| Characteristic | Description |
|----------------|-------------|
| **High‑level** | Abstracts away low‑level details (memory management, hardware access). |
| **Interpreted** | Code is executed by a runtime (the Python interpreter) rather than compiled to native machine code first. |
| **Dynamic typing** | Variable types are determined at runtime; you don’t need to declare them explicitly. |
| **Object‑oriented** | Supports classes, inheritance, polymorphism, and encapsulation. |
| **Multi‑paradigm** | Supports procedural, functional, and object‑oriented styles. |
| **Extensible

# PART 4 -- Runnables & LCEL

# Task 8: Runnables Basics
## 1. Convert chains into Runnables
## 2. Use RunnablePassthrough.
## 3. Compare comonenst using LCEL syntax (|)

In [2]:

from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableLambda
)

# Load API key
load_dotenv()

# 1. Initialize LLM
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7
)

# 2. Create Prompt Template
prompt = PromptTemplate(
    input_variables=["question"],
    template="""
    You are a helpful AI assistant.

    Answer the following question in simple language:

    {question}
    """
)

# 3. Create Runnable Components
parser = StrOutputParser()

# 4. Create a Runnable Chain
chain = prompt | llm | parser

# 5. Use RunnablePassthrough
final_chain = (
    {
        "question": RunnablePassthrough(),
        "original_input": RunnablePassthrough()
    }
    | RunnableLambda(
        lambda x: {
            "question": x["question"],
            "original_input": x["original_input"]
        }
    )
)

# 6. Create Combined Chain
combined_chain = (
    RunnablePassthrough.assign(
        answer=chain
    )
)

# 7. Execute Chain
if __name__ == "__main__":

    question = input("Enter your question: ")

    result = combined_chain.invoke({
        "question": question
    })

    print("\nOriginal Question:")
    print(result["question"])

    print("\nAI Answer:")
    print(result["answer"])



Original Question:
what is DL

AI Answer:
**DL usually stands for “Deep Learning.”**  

- **What it is:** A type of computer program that learns to recognize patterns (like pictures, speech, or text) by looking at lots of examples.  
- **How it works:** It uses many layers of simple “neurons” (tiny math units) stacked together—hence the word *deep*—to turn raw data into useful information.  
- **Why it’s useful:** It can do things like identify faces in photos, translate languages, suggest movies, drive cars, and much more, often better than older methods.  

**Other common meanings of “DL” (just in case):**  

- **Download:** Getting a file from the internet to your device.  
- **Distance Learning:** Studying remotely, often online.  
- **Digital Library:** An online collection of books, articles, or other media.  

When people talk about AI or machine‑learning, “DL” almost always means **Deep Learning**.


# Task 9: LCEL - Based RAG Chain
## . Build an LCEL pipeline:
## . Retriever | Prompt| LLM | Output Parer
## . Test with multiple user queries.

In [4]:

import os
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Load API key
load_dotenv()

# 1. Initialize LLM
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

# 2. Create Sample Documents
documents = [
    Document(page_content="Python is a high-level programming language used for AI, ML and web development."),
    Document(page_content="Machine Learning is a subset of AI that allows computers to learn from data."),
    Document(page_content="Deep Learning uses neural networks to solve complex problems."),
    Document(page_content="LangChain is a framework for building applications using Large Language Models."),
    Document(page_content="RAG stands for Retrieval Augmented Generation. It retrieves relevant documents before generating answers."),
    Document(page_content="ChromaDB is a vector database used to store and search embeddings.")
]

# 3. Create Embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# 4. Create Vector Database
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings
)

# 5. Create Retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

# 6. Create Prompt Template
prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant.

Answer the question using the following context.

Context:
{context}

Question:
{question}

If the answer is not available in the context,
say that you don't have enough information.
""")

# 7. Format Retrieved Documents
def format_docs(docs):
    return "\n\n".join(
        doc.page_content for doc in docs
    )

# 8. Build LCEL RAG Chain
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# 9. Test with Multiple Queries
if __name__ == "__main__":

    questions = [
        "What is Python?",
        "Explain Machine Learning.",
        "What is LangChain?",
        "What is RAG?",
        "What is ChromaDB?"
    ]

    for question in questions:

        print("\nQuestion:", question)

        response = rag_chain.invoke(question)

        print("Answer:", response)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3801.80it/s]



Question: What is Python?
Answer: Python is a high-level programming language that is used for AI, machine learning, and web development.

Question: Explain Machine Learning.
Answer: Machine Learning is a branch of artificial intelligence (AI) that enables computers to automatically improve their performance on a task by learning from data, rather than being explicitly programmed for every possible scenario. In other words, it allows systems to discover patterns, make predictions, or take actions based on the information they have been given, continuously refining their models as more data becomes available.

Question: What is LangChain?
Answer: LangChain is a framework for building applications that use Large Language Models.

Question: What is RAG?
Answer: RAG stands for **Retrieval‑Augmented Generation**. It is an approach that first retrieves relevant documents (or pieces of information) and then uses those retrieved texts to help generate a more informed answer.

Question: What i

# Task 10: Observations & Insights
##  Write short answers:
## 1. Why structured output is important
## 2. Advantages of LCEl over Traditional chains
## 3. When to use parallel vs conditional chains

# 1. Why Structured Output is Important
## .Structured output provides data in a fixed format, such as JSON.
## .It makes LLM responses easy to read, validate and process.
## .Pydantic ensures that output follows the required schema.
## .It improves reliability and integration with applications.

# 2. Advantages of LCEL over Traditional Chains
## . LCEL makes it easy to connect components using the | operator.
## . It supports parallel and conditional execution.
## . It provides reusable and modular pipelines.
## . It supports streaming and asynchronous execution.
## . It makes code simpler, cleaner and easier to maintain.

# 3. When to Use Parallel vs Conditional Chains
## Parallel Chains	Conditional Chains
### Run multiple chains simultaneously.	Select one chain based on a condition.
### Useful for multiple outputs.	Useful for different processing paths.
### Saves execution time when tasks are independent.	Avoids unnecessary processing.
### Example: Answer, summary and follow-up questions.	Example: RAG for factual questions and direct LLM for creative questions.